<a href="https://colab.research.google.com/github/obedglanson/senior_project_slc6a14/blob/main/blind_set_3D.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
!pip install rdkit
!apt-get install openbabel

Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
The following additional packages will be installed:
  libboost-iostreams1.74.0 libinchi1 libmaeparser1 libopenbabel7
The following NEW packages will be installed:
  libboost-iostreams1.74.0 libinchi1 libmaeparser1 libopenbabel7 openbabel
0 upgraded, 5 newly installed, 0 to remove and 53 not upgraded.
Need to get 4,148 kB of archives.
After this operation, 19.1 MB of additional disk space will be used.
Get:1 http://archive.ubuntu.com/ubuntu jammy/main amd64 libboost-iostreams1.74.0 amd64 1.74.0-14ubuntu3 [245 kB]
Get:2 http://archive.ubuntu.com/ubuntu jammy/universe amd64 libinchi1 amd64 1.03+dfsg-4 [455 kB]
Get:3 http://archive.ubuntu.com/ubuntu jammy/universe amd64 libmaeparser1 amd64 1.2.4-1build1 [88.2 kB]
Get:4 http://archive.ubuntu.com/ubuntu jammy/universe amd64 libopenbabel7 amd64 3.1.1+dfsg-6ubuntu5 [3,231 kB]
Get:5 http://archive.ubuntu.com/ubuntu jammy/universe amd64 openbabel am

In [4]:
import pandas as pd

# Load your exact uploaded file
df = pd.read_csv("blindset_raw.csv")

# Write out as a strict OpenBabel .smi file (SMILES [tab] ID)
with open("blindset_raw.smi", "w") as f:
    for index, row in df.iterrows():
        smiles = str(row['Smiles']).strip()
        mol_id = str(row['ID']).strip()
        # Skip empty rows
        if smiles != 'nan':
            f.write(f"{smiles}\t{mol_id}\n")

print("Successfully generated blindset_raw.smi")

Successfully generated blindset_raw.smi


In [6]:
!obabel -ismi blindset_raw.smi -osmi -O blindset_ph.smi -p 7.4

64 molecules converted


In [13]:
import pandas as pd
from rdkit import Chem
from rdkit.Chem import AllChem

# 1. Load the  blind set
smi_path = "blindset_ph.smi"
# Tell pandas it is separated by tabs, and manually assign the column names
df = pd.read_csv(smi_path, sep="\t", header=None, names=["Smiles", "ID"])

# 2. Setup the SDWriter for the unbundled conformers
writer = Chem.SDWriter("BlindSet_3D.sdf")
successful_mols = 0

for index, row in df.iterrows():
    smiles = row['Smiles']
    mol_id = str(row['ID'])

    # Generate 2D graph
    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        print(f"Failed to parse SMILES for ID {mol_id}")
        continue

    # Add explicit hydrogens
    mol = Chem.AddHs(mol)

    # ETKDGv3 Embedding Setup
    params = AllChem.ETKDGv3()
    params.randomSeed = 42
    params.enforceChirality = True

    # Generate 5 conformers per unique molecule
    conf_ids = AllChem.EmbedMultipleConfs(mol, numConfs=5, params=params)

    # Force Field Optimization and Spatial Isolation
    for cid in conf_ids:
        # MMFF94 Optimization with 500 max iterations
        try:
            opt_status = AllChem.MMFFOptimizeMolecule(mol, confId=cid, maxIters=500)

            # Catch silent RDKit forcefield failures
            if opt_status == -1:
                print(f"Missing FF parameters for ID {mol_id}, Conf {cid}. Skipping.")
                continue
            elif opt_status == 1:
                print(f"ID {mol_id}, Conf {cid} did not converge in 500 iterations.")

        except Exception as e:
            print(f"Hard crash during optimization for ID {mol_id}, Conf {cid}: {e}")
            continue

        # Unbundling algorithm: isolate each conformer into a clean object
        single_conf_mol = Chem.Mol(mol)
        single_conf_mol.RemoveAllConformers()

        # Create a detached copy of the conformer
        conf_copy = Chem.Conformer(mol.GetConformer(cid))
        conf_copy.SetId(0) # Force index to 0 for safe SDWriter export

        single_conf_mol.AddConformer(conf_copy, assignId=False)

        # Assign unique, distinct names to each conformer
        single_conf_mol.SetProp("_Name", f"Compound_{mol_id}_conf_{cid}")
        single_conf_mol.SetProp("Parent_ID", mol_id)

        # Save to SDF
        writer.write(single_conf_mol)
        successful_mols += 1

writer.close()
print(f"3D Embedding and Optimization Complete. {successful_mols} geometries saved to BlindSet_3D.sdf")

ID 116, Conf 1 did not converge in 500 iterations.
ID 138, Conf 2 did not converge in 500 iterations.
ID 140, Conf 3 did not converge in 500 iterations.
ID 141, Conf 0 did not converge in 500 iterations.
ID 177, Conf 3 did not converge in 500 iterations.
3D Embedding and Optimization Complete. 320 geometries saved to BlindSet_3D.sdf
